# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [28]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV

In [16]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


## Cleaning from previous lab

In [17]:
spaceship = spaceship.drop(columns = ["Name"])
spaceship = spaceship.drop(columns = ["PassengerId"])
spaceship = spaceship.dropna()
spaceship.isna().sum()

HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Transported     0
dtype: int64

### Cabin

In [18]:
spaceship = spaceship.rename(columns={"Cabin": "cabin_old"})
spaceship["cabin"] = spaceship["cabin_old"].str[0]
spaceship["cabin"].value_counts().sort_index()
spaceship = spaceship.drop(columns = ["cabin_old"])

#Create the training categories
spaceship_cabin_train = pd.DataFrame({"cabin":["A","B","C","D","E","F","G","T"]})

#Create the encoder
encoder_cabin = OneHotEncoder(sparse_output = False)

#Fit the encoder
encoder_cabin.fit(spaceship_cabin_train[["cabin"]])

#transform the data
encoded_cabin = encoder_cabin.transform(spaceship[["cabin"]])

#convert into dataframe
encoded_df_cabin = pd.DataFrame(
    encoded_cabin,
    columns=encoder_cabin.get_feature_names_out(["cabin"]),
    index=spaceship.index
)

#replace the original column
spaceship = pd.concat(
    [spaceship.drop(columns=["cabin"]), encoded_df_cabin],
    axis=1
)
spaceship.head()

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,cabin_A,cabin_B,cabin_C,cabin_D,cabin_E,cabin_F,cabin_G,cabin_T
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


### HomePlanet

In [19]:
spaceship["HomePlanet"].value_counts()

#Create the training categories
spaceship_HomePlanet_train = pd.DataFrame({"HomePlanet":["Earth","Europa","Mars"]})

#Create the encoder
encoder_HomePlanet = OneHotEncoder(sparse_output = False)

#Fit the encoder
encoder_HomePlanet.fit(spaceship_HomePlanet_train[["HomePlanet"]])

#transform the data
encoded_HomePlanet = encoder_HomePlanet.transform(spaceship[["HomePlanet"]])

#convert into dataframe
encoded_df_HomePlanet = pd.DataFrame(
    encoded_HomePlanet,
    columns=encoder_HomePlanet.get_feature_names_out(["HomePlanet"]),
    index=spaceship.index
)

#replace the original column
spaceship = pd.concat(
    [spaceship.drop(columns=["HomePlanet"]), encoded_df_HomePlanet],
    axis=1
)
spaceship.head()

,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,...,cabin_B,cabin_C,cabin_D,cabin_E,cabin_F,cabin_G,cabin_T,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars
0,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


### Destination

In [20]:
spaceship["Destination"].value_counts()

#Create the training categories
spaceship_Destination_train = pd.DataFrame({"Destination":["TRAPPIST-1e","55 Cancri e","PSO J318.5-22"]})

#Create the encoder
encoder_Destination = OneHotEncoder(sparse_output = False)

#Fit the encoder
encoder_Destination.fit(spaceship_Destination_train[["Destination"]])

#transform the data
encoded_Destination = encoder_Destination.transform(spaceship[["Destination"]])

#convert into dataframe
encoded_df_Destination = pd.DataFrame(
    encoded_Destination,
    columns=encoder_Destination.get_feature_names_out(["Destination"]),
    index=spaceship.index
)

#replace the original column
spaceship = pd.concat(
    [spaceship.drop(columns=["Destination"]), encoded_df_Destination],
    axis=1
)
spaceship.head()

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,cabin_A,...,cabin_E,cabin_F,cabin_G,cabin_T,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e
0,False,39.0,False,0.0,0.0,0.0,0.0,0.0,False,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,False,24.0,False,109.0,9.0,25.0,549.0,44.0,True,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,False,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,False,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,False,16.0,False,303.0,70.0,151.0,565.0,2.0,True,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


### CryoSleep, VIP, Transported (booleans)

In [21]:
spaceship['CryoSleep'] = spaceship['CryoSleep'].astype(int)
spaceship['VIP'] = spaceship['VIP'].astype(int)
spaceship['Transported'] = spaceship['Transported'].astype(int)
spaceship.head()

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,cabin_A,...,cabin_E,cabin_F,cabin_G,cabin_T,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e
0,0,39.0,0,0.0,0.0,0.0,0.0,0.0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,0,24.0,0,109.0,9.0,25.0,549.0,44.0,1,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,0,58.0,1,43.0,3576.0,0.0,6715.0,49.0,0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,0,33.0,0,0.0,1283.0,371.0,3329.0,193.0,0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,0,16.0,0,303.0,70.0,151.0,565.0,2.0,1,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

## Test Split and Model Run

### Train Test Split

In [22]:
features = spaceship.drop(columns = ["Transported"])
target = spaceship["Transported"]

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.20, random_state=17)

normalizer = MinMaxScaler()

#Remember than MinMAxScaler only workds with numeric columns, therefore 
normalizer.fit(X_train)

# Apply the transformation, and the result (gives an array) transform to a dataframe
X_train_norm = normalizer.transform(X_train)
X_train_norm = pd.DataFrame(X_train_norm,columns=X_train.columns)

X_test_norm = normalizer.transform(X_test)
X_test_norm = pd.DataFrame(X_test_norm,columns = X_test.columns)
X_test_norm.describe()


,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,cabin_A,cabin_B,...,cabin_E,cabin_F,cabin_G,cabin_T,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e
count,1353.000000,1353.000000,1353.000000,1353.000000,1353.000000,1353.000000,1353.000000,1353.000000,1353.000000,1353.000000,...,1353.00000,1353.000000,1353.000000,1353.000000,1353.000000,1353.000000,1353.000000,1353.000000,1353.000000,1353.000000
mean,0.365115,0.378390,0.031781,0.020864,0.014410,0.008152,0.016277,0.013999,0.028825,0.113082,...,0.09830,0.323725,0.294161,0.000739,0.519586,0.263858,0.216556,0.223947,0.100517,0.675536
std,0.481640,0.192240,0.175482,0.059244,0.047719,0.025787,0.055525,0.051409,0.167376,0.316810,...,0.29783,0.468070,0.455833,0.027186,0.499801,0.440886,0.412050,0.417041,0.300800,0.468347
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.253165,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.354430,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.493671,0.000000,0.003831,0.002683,0.001320,0.003526,0.002065,0.000000,0.000000,...,0.00000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000
max,1.000000,1.000000,1.000000,0.655040,0.602355,0.455687,0.740539,0.712284,1.000000,1.000000,...,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


### Model

- Evaluate your model

In [25]:
# Initialize a Bagging regressor
bagging_reg = BaggingRegressor(DecisionTreeRegressor(max_depth=20),
                               n_estimators=100,
                               max_samples = 1000)

# Training the model with normalized data
bagging_reg.fit(X_train_norm, y_train)

# Evaluate model's performance
pred = bagging_reg.predict(X_test_norm)

print("MAE", mean_absolute_error(pred, y_test))
print("RMSE", mean_squared_error(pred, y_test))
print("R2 score", bagging_reg.score(X_test_norm, y_test))

MAE 0.279682589673104
RMSE 0.14474564887304425
R2 score 0.42037623092439913


## Hyper-Tuning

**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [26]:
grid = {"n_estimators": [50, 100, 200],
        "estimator__max_leaf_nodes": [25, 50],
        "estimator__max_depth":[10,30]}

- Run Grid Search

In [29]:
bagging_reg = BaggingRegressor(DecisionTreeRegressor(max_depth=20),
                               n_estimators=100,
                               max_samples = 1000)

#here we will use Grid Search Cross Validation
model = GridSearchCV(estimator = bagging_reg, param_grid = grid, cv=2)

model.fit(X_train_norm, y_train)

# After training, we check what are the best values for the 
# hyperparameters that we have tested.
model.best_params_

# You can retrieve the best model with the best parameters when 
# accessing best_estimator_ attribute
best_model = model.best_estimator_

- Evaluate your model

In [32]:
pred = best_model.predict(X_test_norm)

print("MAE", mean_absolute_error(y_test, pred))
print("RMSE", np.sqrt(mean_squared_error(y_test, pred)))
print("R2 score", best_model.score(X_test_norm, y_test))

MAE 0.282122541813412
RMSE 0.3766006331470951
R2 score 0.4320596028898973
